# Módulo 2 — CNN: Clasificación de Conducción Distractiva
**Transfer Learning con ResNet18**
**Dataset real**: `arafatsahinafridi/multi-class-driver-behavior-image-dataset` (Kaggle)

Universidad Nacional de Colombia · IRNA 2026-01

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pickle, sys, shutil
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt, seaborn as sns
from pathlib import Path

import torch, torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Subset
from torchvision.datasets import ImageFolder
from torch.optim.lr_scheduler import ReduceLROnPlateau
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import (accuracy_score, f1_score, precision_score,
                              recall_score, classification_report, confusion_matrix)
from PIL import Image

SEED=42; np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

EPOCHS=25; BATCH_SIZE=32; LR=1e-4; IMG_SIZE=224; PATIENCE=5
BASE_DIR=Path('.'); MODELS_DIR=BASE_DIR/'models'; MODELS_DIR.mkdir(exist_ok=True)
print(f"Dispositivo: {device} | Épocas máx: {EPOCHS} | LR: {LR}")

## 1. Descarga y localización del dataset real

In [ ]:
import kagglehub

print("Descargando dataset desde Kaggle...")
try:
    raw_path = Path(kagglehub.dataset_download(
        "arafatsahinafridi/multi-class-driver-behavior-image-dataset"))
except Exception as e:
    sys.exit(f"ERROR: {e}\nConfigura ~/.kaggle/kaggle.json")

# Localizar carpeta con clases c0, c1, ...
data_dir = None
for candidate in [raw_path, *raw_path.rglob("train"), *raw_path.rglob("Train")]:
    candidate = Path(candidate)
    if candidate.is_dir():
        cls_dirs = [d for d in candidate.iterdir() if d.is_dir()
                    and d.name.lower().startswith("c") and d.name[1:].isdigit()]
        if len(cls_dirs) >= 5:
            data_dir = candidate; break
if data_dir is None:
    cls_dirs = [d for d in raw_path.iterdir() if d.is_dir()
                and d.name.lower().startswith("c") and d.name[1:].isdigit()]
    if len(cls_dirs) >= 5:
        data_dir = raw_path
assert data_dir, f"No se encontró estructura c0/c1/... en {raw_path}"
print(f"Directorio de clases: {data_dir}")

# Contar imágenes
for d in sorted(data_dir.iterdir(), key=lambda x: x.name):
    if d.is_dir() and d.name.startswith("c"):
        n = len(list(d.glob("*.jpg")) + list(d.glob("*.png")) + list(d.glob("*.jpeg")))
        print(f"  {d.name}: {n:,} imágenes")

## 2. Dataset y DataLoaders

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(0.5),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    transforms.RandomAffine(degrees=0, translate=(0.1,0.1)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
val_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

full_ds = ImageFolder(str(data_dir), transform=val_tf)
class_names = full_ds.classes
num_classes = len(class_names)
print(f"Clases ({num_classes}): {class_names}")
print(f"Total imágenes: {len(full_ds):,}")

targets = np.array(full_ds.targets)
sss = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
train_idx, val_idx = next(sss.split(np.zeros(len(targets)), targets))

train_ds_aug = ImageFolder(str(data_dir), transform=train_tf)
train_subset = Subset(train_ds_aug, train_idx)
val_subset   = Subset(full_ds, val_idx)

train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_subset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
print(f"Train: {len(train_subset):,} | Val: {len(val_subset):,}")

## 3. Modelo: ResNet18 con Transfer Learning

In [ ]:
def build_model(num_classes):
    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    # Descongelar layer3, layer4 y fc para fine-tuning
    for name, param in model.named_parameters():
        param.requires_grad = any(k in name for k in ["layer3","layer4","fc"])
    in_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(0.5),
        nn.Linear(in_features, 256),
        nn.ReLU(inplace=True),
        nn.BatchNorm1d(256),
        nn.Dropout(0.3),
        nn.Linear(256, num_classes),
    )
    return model

model = build_model(num_classes).to(device)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Parámetros entrenables: {trainable:,} / {total:,} ({trainable/total*100:.1f}%)")

## 4. Entrenamiento

In [ ]:
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()), lr=LR, weight_decay=1e-4)
scheduler = ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=3, verbose=True)

history = {"train_loss":[], "val_loss":[], "train_acc":[], "val_acc":[]}
best_val_acc = 0.0; best_state = None; patience_cnt = 0

for ep in range(1, EPOCHS+1):
    # Train
    model.train(); tr_loss=0.0; tr_cor=0; tr_n=0
    for X,y in train_loader:
        X,y=X.to(device),y.to(device); optimizer.zero_grad()
        out=model(X); loss=criterion(out,y); loss.backward(); optimizer.step()
        tr_loss+=loss.item()*len(y); tr_cor+=(out.argmax(1)==y).sum().item(); tr_n+=len(y)
    tr_loss/=tr_n; tr_acc=tr_cor/tr_n
    
    # Val
    model.eval(); va_loss=0.0; va_cor=0; va_n=0
    all_preds=[]; all_tgts=[]
    with torch.no_grad():
        for X,y in val_loader:
            X,y=X.to(device),y.to(device); out=model(X); loss=criterion(out,y)
            va_loss+=loss.item()*len(y); va_cor+=(out.argmax(1)==y).sum().item(); va_n+=len(y)
            all_preds.extend(out.argmax(1).cpu().numpy()); all_tgts.extend(y.cpu().numpy())
    va_loss/=va_n; va_acc=va_cor/va_n
    scheduler.step(va_acc)
    
    history["train_loss"].append(tr_loss); history["val_loss"].append(va_loss)
    history["train_acc"].append(tr_acc);   history["val_acc"].append(va_acc)
    print(f"Ep {ep:3d}/{EPOCHS} | TrLoss={tr_loss:.4f} TrAcc={tr_acc*100:.2f}% | VaLoss={va_loss:.4f} VaAcc={va_acc*100:.2f}%")
    
    if va_acc > best_val_acc:
        best_val_acc=va_acc; best_state={k:v.clone() for k,v in model.state_dict().items()}; patience_cnt=0
    else:
        patience_cnt+=1
        if patience_cnt>=PATIENCE: print(f"Early stop epoch {ep}"); break

model.load_state_dict(best_state)
print(f"\nMejor val accuracy: {best_val_acc*100:.2f}%")

## 5. Curvas de entrenamiento

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
eps = range(1, len(history["train_loss"])+1)
axes[0].plot(eps, history["train_loss"], label="Train", color="#2C5282")
axes[0].plot(eps, history["val_loss"],   label="Val",   color="#E53E3E")
axes[0].set_title("Pérdida"); axes[0].set_xlabel("Época"); axes[0].legend()
axes[1].plot(eps, [v*100 for v in history["train_acc"]], label="Train", color="#2C5282")
axes[1].plot(eps, [v*100 for v in history["val_acc"]],   label="Val",   color="#E53E3E")
axes[1].set_title("Accuracy (%)"); axes[1].set_xlabel("Época"); axes[1].legend()
plt.tight_layout()
plt.savefig("fig_training_curves.png", dpi=120, bbox_inches="tight")
plt.show()
print("Guardado: fig_training_curves.png")

## 6. Evaluación final: métricas completas

In [ ]:
model.eval(); all_preds=[]; all_tgts=[]
with torch.no_grad():
    for X,y in val_loader:
        out=model(X.to(device)); all_preds.extend(out.argmax(1).cpu().numpy()); all_tgts.extend(y.numpy())
all_preds=np.array(all_preds); all_tgts=np.array(all_tgts)

acc_g  = accuracy_score(all_tgts, all_preds)
f1_mac = f1_score(all_tgts, all_preds, average="macro",    zero_division=0)
f1_w   = f1_score(all_tgts, all_preds, average="weighted", zero_division=0)
prec   = precision_score(all_tgts, all_preds, average="macro", zero_division=0)
rec    = recall_score(all_tgts, all_preds,    average="macro", zero_division=0)

print(f"Accuracy global  : {acc_g*100:.2f}%")
print(f"F1 macro         : {f1_mac:.4f}")
print(f"F1 weighted      : {f1_w:.4f}")
print(f"Precision macro  : {prec:.4f}")
print(f"Recall macro     : {rec:.4f}")
print()
print(classification_report(all_tgts, all_preds, target_names=class_names, zero_division=0))

## 7. Matriz de confusión

In [ ]:
cm = confusion_matrix(all_tgts, all_preds)
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=class_names, yticklabels=class_names)
axes[0].set_title("Matriz de Confusión (absoluta)"); axes[0].set_xlabel("Predicho"); axes[0].set_ylabel("Real")
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='YlOrRd', ax=axes[1],
            xticklabels=class_names, yticklabels=class_names)
axes[1].set_title("Matriz de Confusión (normalizada)")
axes[1].set_xlabel("Predicho"); axes[1].set_ylabel("Real")
plt.tight_layout()
plt.savefig("fig_confusion_matrix.png", dpi=120, bbox_inches="tight")
plt.show()
print("Guardado: fig_confusion_matrix.png")

## 8. Ejemplos clasificados correcta e incorrectamente

In [ ]:
import random; random.seed(42)

def show_examples(indices, title, filename, n=16):
    indices = random.sample(list(indices), min(n, len(indices)))
    n_cols=4; n_rows=(len(indices)+3)//4
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols*3, n_rows*3))
    axes = axes.flatten() if n_rows>1 else [axes]*n_cols
    for ax_i, idx in enumerate(indices):
        X, y_true = full_ds[idx]
        X_t = X.unsqueeze(0).to(device)
        with torch.no_grad():
            y_pred = model(X_t).argmax(1).item()
        # Desnormalizar para visualizar
        mean=torch.tensor(IMAGENET_MEAN).view(3,1,1); std=torch.tensor(IMAGENET_STD).view(3,1,1)
        img = (X * std + mean).clamp(0,1).permute(1,2,0).numpy()
        axes[ax_i].imshow(img); axes[ax_i].axis("off")
        color = "#48BB78" if y_pred==y_true else "#E53E3E"
        axes[ax_i].set_title(f"V:{class_names[y_true]}\nP:{class_names[y_pred]}",
                             fontsize=7, color=color)
    for ax in axes[len(indices):]: ax.axis("off")
    plt.suptitle(title, fontsize=12); plt.tight_layout()
    plt.savefig(filename, dpi=100, bbox_inches="tight"); plt.show()
    print(f"Guardado: {filename}")

correct_idx   = [i for i, (p,t) in enumerate(zip(all_preds, all_tgts)) if p==t]
incorrect_idx = [i for i, (p,t) in enumerate(zip(all_preds, all_tgts)) if p!=t]

# Mapear índices de val_subset a índices de full_ds
val_full_idx = val_subset.indices
correct_full   = [val_full_idx[i] for i in correct_idx]
incorrect_full = [val_full_idx[i] for i in incorrect_idx]

show_examples(correct_full,   "Clasificadas CORRECTAMENTE ✓", "fig_correct_examples.png")
show_examples(incorrect_full, "Clasificadas INCORRECTAMENTE ✗", "fig_wrong_examples.png")

## 9. F1 por clase y análisis de distracciones

In [ ]:
f1_per_class = f1_score(all_tgts, all_preds, average=None, zero_division=0)
acc_per_class = cm.diagonal() / cm.sum(axis=1)

PREVENTIVE = {
    "c0": "Conducción segura. Mantener buenas prácticas.",
    "c1": "Instalar bloqueador de teléfono automático al conducir.",
    "c2": "Obligar uso de manos libres o Bluetooth vehicular.",
    "c3": "Política de tolerancia cero para uso de teléfono.",
    "c4": "Sistema de llamadas integrado al vehículo.",
    "c5": "Capacitar en manejo de controles sin desviar la vista.",
    "c6": "Prohibir consumo de bebidas al conducir.",
    "c7": "Asegurar todos los objetos antes de iniciar la marcha.",
    "c8": "Política de cero arreglo personal en movimiento.",
    "c9": "Capacitar en comunicación segura con pasajeros.",
}
print("\nAnálisis por clase:")
df_cls = []
for i, cls in enumerate(class_names):
    df_cls.append({'Clase':cls, 'F1':f1_per_class[i], 'Acc %':acc_per_class[i]*100,
                   'Medida preventiva':PREVENTIVE.get(cls,'Revisar comportamiento.')})
import pandas as pd
df_cls = pd.DataFrame(df_cls).sort_values('F1', ascending=True)
print(df_cls[['Clase','F1','Acc %']].to_string(index=False))
print(f"\n3 clases más difíciles: {df_cls.head(3)['Clase'].tolist()}")

fig, ax = plt.subplots(figsize=(10,5))
colors = ['#E53E3E' if f<0.7 else '#ECC94B' if f<0.85 else '#48BB78' for f in df_cls['F1']]
ax.barh(df_cls['Clase'], df_cls['F1'], color=colors)
ax.axvline(0.7, color='orange', linestyle='--', linewidth=1, label='Umbral 0.70')
ax.axvline(0.85, color='green', linestyle='--', linewidth=1, label='Umbral 0.85')
ax.set_xlabel("F1-score"); ax.set_title("F1 por clase"); ax.legend()
plt.tight_layout()
plt.savefig("fig_f1_per_class.png", dpi=120, bbox_inches="tight"); plt.show()

## 10. Guardar artefactos

In [ ]:
model_data = {"state_dict": model.state_dict(),
               "num_classes": num_classes, "class_names": class_names}
metrics_data = {"accuracy": float(acc_g), "f1_macro": float(f1_mac),
                "f1_weighted": float(f1_w), "precision_macro": float(prec),
                "recall_macro": float(rec), "best_val_acc": float(best_val_acc),
                "num_classes": num_classes, "class_names": class_names,
                "f1_per_class": {c:float(f) for c,f in zip(class_names, f1_per_class)},
                "history": history}

import torch; import pickle
for out_dir in [MODELS_DIR, Path('../webapp/models')]:
    out_dir.mkdir(exist_ok=True)
    torch.save(model_data, out_dir/"resnet18_driver.pt")
    with open(out_dir/"class_names.pkl", "wb") as f: pickle.dump(class_names, f)
    with open(out_dir/"cnn_metrics.pkl", "wb") as f: pickle.dump(metrics_data, f)
    print(f"Guardado en {out_dir}: resnet18_driver.pt | class_names.pkl | cnn_metrics.pkl")

print(f"\n[OK] Accuracy: {acc_g*100:.2f}% | F1 macro: {f1_mac:.4f}")
print(f"     Clases: {class_names}")